# Data Cleaning — Rossmann Store Sales
Lanjutan dari `Data_Understanding.ipynb`.

**Prinsip di notebook ini:**
- Cleaning dilakukan **terpisah per dataset** (`train` dan `store`), tidak digabung (merge) dulu.
- Tidak membuat kolom/fitur baru (feature engineering) di tahap ini — itu akan dilakukan di notebook `03_Feature_Engineering.ipynb`.
- Merge antar dataset juga akan dilakukan nanti, saat Feature Engineering / ETL / Load ke warehouse — bukan di sini.
- Data mentah tetap dijaga utuh: semua proses cleaning dilakukan pada hasil `.copy()`, bukan pada `train`/`store` asli.
- **Missing value tidak selalu berarti data hilang.** Sebelum memutuskan imputasi, kita cek dulu apakah pola-nya *structural/business missing* (nilai kosong karena memang tidak berlaku), atau *missing* murni karena data tidak tercatat. Hanya yang benar-benar missing yang diimputasi; yang structural dibiarkan apa adanya.

In [90]:
import pandas as pd
import numpy as np
import os
import shutil

## 1. Load Data Mentah (Raw)
Pada `Data_Understanding.ipynb`, muncul `DtypeWarning` saat membaca `train.csv` karena kolom `StateHoliday` berisi campuran tipe (string `'0'` dan angka `0`). Di sini kita tetapkan `dtype` secara eksplisit untuk `StateHoliday` supaya tidak ada warning dan tipe datanya konsisten sejak awal dibaca.

In [91]:
train = pd.read_csv(
    'rossmann-store-sales/train.csv',
    dtype={'StateHoliday': str}
)
store = pd.read_csv('rossmann-store-sales/store.csv')

## 2. Buat Salinan (Copy) Dataset untuk Dibersihkan
`train` dan `store` asli **tidak disentuh** mulai dari sini. Semua perubahan hanya dilakukan pada `train_clean` dan `store_clean`, dan keduanya dibersihkan **secara independen** — tidak di-merge di tahap ini.

In [92]:
train_clean = train.copy()
store_clean = store.copy()

print('train_clean shape:', train_clean.shape)
print('store_clean shape:', store_clean.shape)

train_clean shape: (1017209, 9)
store_clean shape: (1115, 10)


## 3. Cleaning `train_clean`

### 3.1 Parse kolom `Date` ke tipe datetime
Dari data understanding, `Date` terbaca sebagai string dengan 942 nilai unik, rentang 2013-01-01 s.d. 2015-07-31. Kita ubah ke tipe `datetime` supaya bisa diproses sebagai tanggal (bukan cleaning nilai, hanya koreksi tipe data).

In [93]:
train_clean['Date'] = pd.to_datetime(train_clean['Date'], format='%Y-%m-%d')
print('Date dtype after parsing:', train_clean['Date'].dtype)
print('Date min:', train_clean['Date'].min())
print('Date max:', train_clean['Date'].max())

Date dtype after parsing: datetime64[us]
Date min: 2013-01-01 00:00:00
Date max: 2015-07-31 00:00:00


### 3.2 Standardisasi nilai `StateHoliday`
Data understanding menunjukkan `StateHoliday` punya 5 kategori unik: `'0'` (855,087 baris), `0` (131,072 baris, tipe berbeda dari `'0'` akibat isu parsing), `'a'` (20,260), `'b'` (6,690), `'c'` (4,100). Nilai `'0'` dan `0` sebenarnya merepresentasikan hal yang sama (tidak ada libur), jadi digabung. Label lain diberi nama yang lebih jelas supaya mudah dibaca, tanpa mengubah maknanya.

In [94]:
print('Before cleaning:', train_clean['StateHoliday'].unique())
print(train_clean['StateHoliday'].value_counts())

Before cleaning: <StringArray>
['0', 'a', 'b', 'c']
Length: 4, dtype: str
StateHoliday
0    986159
a     20260
b      6690
c      4100
Name: count, dtype: int64


In [95]:
state_holiday_map = {
    '0': 'NoHoliday',
    'a': 'PublicHoliday',
    'b': 'EasterHoliday',
    'c': 'Christmas'
}
train_clean['StateHoliday'] = train_clean['StateHoliday'].map(state_holiday_map)

print('After cleaning:', train_clean['StateHoliday'].unique())
print(train_clean['StateHoliday'].value_counts())

After cleaning: <StringArray>
['NoHoliday', 'PublicHoliday', 'EasterHoliday', 'Christmas']
Length: 4, dtype: str
StateHoliday
NoHoliday        986159
PublicHoliday     20260
EasterHoliday      6690
Christmas          4100
Name: count, dtype: int64


### 4.3 Cek duplikat
Data understanding sudah menunjukkan 0 baris duplikat di `train`. Kita verifikasi ulang di `train_clean` sebagai langkah validasi, tanpa perlu tindakan lebih lanjut.

In [96]:
dup_count = train_clean.duplicated().sum()
print('Duplicated rows:', dup_count)

Duplicated rows: 0


### 4.4 Validasi ulang `train_clean` setelah cleaning

In [97]:
print(train_clean.info())
print()
print('Null values:')
print(train_clean.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype         
---  ------         --------------    -----         
 0   Store          1017209 non-null  int64         
 1   DayOfWeek      1017209 non-null  int64         
 2   Date           1017209 non-null  datetime64[us]
 3   Sales          1017209 non-null  int64         
 4   Customers      1017209 non-null  int64         
 5   Open           1017209 non-null  int64         
 6   Promo          1017209 non-null  int64         
 7   StateHoliday   1017209 non-null  str           
 8   SchoolHoliday  1017209 non-null  int64         
dtypes: datetime64[us](1), int64(7), str(1)
memory usage: 69.8 MB
None

Null values:
Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64


## 5. Cleaning `store_clean`

### 5.1 Missing value sebelum cleaning
Dari data understanding: `CompetitionDistance` kosong 3 baris, `CompetitionOpenSinceMonth`/`CompetitionOpenSinceYear` kosong masing-masing 354 baris, dan `Promo2SinceWeek`/`Promo2SinceYear`/`PromoInterval` kosong masing-masing 544 baris (dari total 1,115 toko).

In [98]:
print('Missing values before cleaning:')
print(store_clean.isnull().sum())

Missing values before cleaning:
Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64


### 5.2 Cek pola missing pada `Promo2SinceWeek`, `Promo2SinceYear`, `PromoInterval`
`Promo2` punya 2 nilai: `1` (571 toko) dan `0` (544 toko). Jumlah missing di `Promo2SinceWeek`, `Promo2SinceYear`, dan `PromoInterval` **persis 544** — sama persis dengan jumlah toko `Promo2 = 0`. Kita cross-check polanya berikut ini.

In [99]:
print('Promo2 value counts:')
print(store_clean['Promo2'].value_counts())
print()
print('Missing Promo2SinceWeek, dipecah berdasarkan Promo2:')
print(store_clean.groupby('Promo2')['Promo2SinceWeek'].apply(lambda x: x.isnull().sum()))

Promo2 value counts:
Promo2
1    571
0    544
Name: count, dtype: int64

Missing Promo2SinceWeek, dipecah berdasarkan Promo2:
Promo2
0    544
1      0
Name: Promo2SinceWeek, dtype: int64


**Kesimpulan:** semua baris dengan `Promo2SinceWeek`/`Promo2SinceYear`/`PromoInterval` kosong adalah toko dengan `Promo2 = 0` (toko tidak ikut program Promo2). Artinya nilai kosong ini bukan data hilang, melainkan **structural missing / business missing** — memang tidak berlaku untuk toko yang tidak ikut Promo2.

➡️ **Keputusan: dibiarkan (tidak dihapus, tidak diimputasi).** Interpretasi/encoding untuk kebutuhan modeling akan ditangani nanti di tahap Feature Engineering.

In [100]:
# Tidak ada perubahan pada Promo2SinceWeek, Promo2SinceYear, PromoInterval
print('Promo2SinceWeek missing (dibiarkan):', store_clean['Promo2SinceWeek'].isnull().sum())
print('Promo2SinceYear missing (dibiarkan):', store_clean['Promo2SinceYear'].isnull().sum())
print('PromoInterval missing (dibiarkan):', store_clean['PromoInterval'].isnull().sum())

Promo2SinceWeek missing (dibiarkan): 544
Promo2SinceYear missing (dibiarkan): 544
PromoInterval missing (dibiarkan): 544


### 5.3 Cek pola missing pada `CompetitionOpenSinceMonth` &amp; `CompetitionOpenSinceYear`
Sebelum memutuskan, kita cek apakah missing pada kedua kolom ini berhubungan dengan `CompetitionDistance` — misalnya, toko tanpa kompetitor di sekitar wajar tidak punya info kapan kompetitor mulai beroperasi.

In [101]:
store_clean[
    store_clean['CompetitionOpenSinceMonth'].isna()
][['CompetitionDistance']].describe()

,CompetitionDistance
count,351.000000
mean,5430.854701
std,7407.075757
min,20.000000
25%,630.000000
50%,2460.000000
75%,7900.000000
max,75860.000000


In [102]:
print('CompetitionDistance untuk baris dengan CompetitionOpenSinceMonth missing:')
print(store_clean[store_clean['CompetitionOpenSinceMonth'].isna()]['CompetitionDistance'].describe())
print()
print('CompetitionDistance untuk baris dengan CompetitionOpenSinceMonth TERISI:')
print(store_clean[store_clean['CompetitionOpenSinceMonth'].notna()]['CompetitionDistance'].describe())

CompetitionDistance untuk baris dengan CompetitionOpenSinceMonth missing:
count      351.000000
mean      5430.854701
std       7407.075757
min         20.000000
25%        630.000000
50%       2460.000000
75%       7900.000000
max      75860.000000
Name: CompetitionDistance, dtype: float64

CompetitionDistance untuk baris dengan CompetitionOpenSinceMonth TERISI:
count      761.000000
mean      5392.930355
std       7783.215981
min         30.000000
25%        730.000000
50%       2230.000000
75%       6470.000000
max      58260.000000
Name: CompetitionDistance, dtype: float64


**Catatan:** `CompetitionDistance` tetap terisi pada sebagian besar baris yang `CompetitionOpenSinceMonth`-nya kosong — artinya toko tersebut **memang punya kompetitor** (ada jaraknya), hanya saja tanggal kompetitor mulai beroperasi tidak tercatat. Ini kemungkinan besar bersifat *missing* (data tidak tercatat), bukan murni *not applicable* seperti kasus Promo2. Namun karena menebak bulan/tahun spesifik berisiko menyesatkan analisis, dan interpretasi kolom ini (mis. dikonversi jadi 'lama kompetitor beroperasi') lebih tepat dilakukan setelah fitur turunannya jelas, keputusan untuk saat ini:

➡️ **Keputusan: dibiarkan dulu (belum dihapus, belum diimputasi).** Akan ditangani saat Feature Engineering, setelah jelas fitur apa yang akan diturunkan dari kolom ini.

In [103]:
# Tidak ada perubahan pada CompetitionOpenSinceMonth, CompetitionOpenSinceYear
print('CompetitionOpenSinceMonth missing (dibiarkan):', store_clean['CompetitionOpenSinceMonth'].isnull().sum())
print('CompetitionOpenSinceYear missing (dibiarkan):', store_clean['CompetitionOpenSinceYear'].isnull().sum())

CompetitionOpenSinceMonth missing (dibiarkan): 354
CompetitionOpenSinceYear missing (dibiarkan): 354


### 5.4 Imputasi `CompetitionDistance`
Missing values pada `CompetitionDistance` hanya berjumlah 3 dari 1,115 toko (0.27%) — sangat kecil dan tidak menunjukkan pola structural seperti dua kasus di atas. Untuk kasus missing yang murni dan kecil seperti ini, pilihan yang umum di industri adalah imputasi dengan **median**, karena median tidak terlalu dipengaruhi outlier dibanding mean atau nilai maksimum.

In [104]:
median_distance = store_clean['CompetitionDistance'].median()
print('Median CompetitionDistance:', median_distance)

store_clean['CompetitionDistance'] = store_clean['CompetitionDistance'].fillna(median_distance)
print('Missing setelah imputasi:', store_clean['CompetitionDistance'].isnull().sum())

Median CompetitionDistance: 2325.0
Missing setelah imputasi: 0


### 5.5 Cek duplikat

In [105]:
dup_store = store_clean.duplicated().sum()
print('Duplicated rows:', dup_store)

Duplicated rows: 0


### 5.6 Validasi ulang `store_clean` setelah cleaning

In [106]:
print(store_clean.info())
print()
print('Null values setelah cleaning:')
print(store_clean.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115 non-null   str    
 2   Assortment                 1115 non-null   str    
 3   CompetitionDistance        1115 non-null   float64
 4   CompetitionOpenSinceMonth  761 non-null    float64
 5   CompetitionOpenSinceYear   761 non-null    float64
 6   Promo2                     1115 non-null   int64  
 7   Promo2SinceWeek            571 non-null    float64
 8   Promo2SinceYear            571 non-null    float64
 9   PromoInterval              571 non-null    str    
dtypes: float64(5), int64(2), str(3)
memory usage: 87.2 KB
None

Null values setelah cleaning:
Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            0
Compe

**Ringkasan keputusan cleaning `store_clean`:**

| Kolom | Keputusan |
|---|---|
| `CompetitionDistance` | Diimputasi (median) |
| `CompetitionOpenSinceMonth` | Dibiarkan (ditangani saat Feature Engineering) |
| `CompetitionOpenSinceYear` | Dibiarkan (ditangani saat Feature Engineering) |
| `Promo2SinceWeek` | Dibiarkan (structural missing, Promo2=0) |
| `Promo2SinceYear` | Dibiarkan (structural missing, Promo2=0) |
| `PromoInterval` | Dibiarkan (structural missing, Promo2=0) |

## 6. Simpan Dataset yang Sudah Bersih
Disimpan **terpisah per dataset** (tidak di-merge) ke folder `cleaned/`. Merge antara `train_clean` dan `store_clean` sengaja **tidak** dilakukan di sini — itu akan dilakukan nanti di tahap Feature Engineering, ETL, atau saat Load ke data warehouse, supaya setiap tahap punya tanggung jawab yang jelas.

In [107]:
cleaned_dir = 'cleaned'
os.makedirs(cleaned_dir, exist_ok=True)

train_clean.to_csv(f'{cleaned_dir}/train_clean.csv', index=False)
store_clean.to_csv(f'{cleaned_dir}/store_clean.csv', index=False)

print('Cleaned files saved to:', cleaned_dir)
print(os.listdir(cleaned_dir))

Cleaned files saved to: cleaned
['store_clean.csv', 'train_clean.csv']



## 7. Conclusion

The data cleaning process focused on improving data quality while preserving business meaning.

The Date column was converted to datetime, inconsistent StateHoliday values were standardized, and CompetitionDistance missing values were imputed using the median due to their low proportion.

Missing values related to Promo2 were intentionally retained because they represent structural missing values (stores not participating in the Promo2 program).

CompetitionOpenSinceMonth and CompetitionOpenSinceYear were also retained because the missing values likely indicate unavailable historical information rather than data entry errors.

The cleaned datasets were exported separately as train_clean.csv and store_clean.csv for the next stage of feature engineering.